# Initialization

Files for case studies, sample points, and meteorological station data are available on [GitHub](https://github.com/thomaslu678/Praxis-Lab-25-26/tree/main/assets). These assets should be uploaded to an individual Google Earth Engine project for further access.

## Expected Feature Columns
### Studies

* name (str)
* start (int)
* end (int)
* similarity (float)
* station (str)

### Points

* name (str)
* distance (int)

### Meterological Data

* STATION (str)
* Year (int)
* Month (int)
* Day (int)
* Hour (int)
* temperature (float)
* relative_humidity (float)

In [1]:
import ee, requests, geemap.core
import pandas as pd
from google.colab import files

project_name = "gee-481701"
studies_path = "projects/gee-481701/assets/studies"
points_path = "projects/gee-481701/assets/points"
met_path = "projects/gee-481701/assets/met_data"

export_path = "projects/gee-481701/assets/samples"


ee.Authenticate()
ee.Initialize(project = project_name)

studies = ee.FeatureCollection(studies_path)
points = ee.FeatureCollection(points_path)
met_data = ee.FeatureCollection(met_path)

# Combine Landsat data into single dataset

Includes Landsat 9, 8, 7, 5, and 4

In [2]:
ls9: ee.ImageCollection = ee.ImageCollection("LANDSAT/LC09/C02/T1_L2")
ls9 = ls9.select([
    "SR_B2", "SR_B3", "SR_B4", "SR_B5", "SR_B6", "SR_B7", "ST_B10", "QA_PIXEL", "QA_RADSAT"
], [
    "blue", "green", "red", "nir", "swir1", "swir2", "lst", "QA_PIXEL", "QA_RADSAT"
])

ls8: ee.ImageCollection = ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
ls8 = ls8.select([
    "SR_B2", "SR_B3", "SR_B4", "SR_B5", "SR_B6", "SR_B7", "ST_B10", "QA_PIXEL", "QA_RADSAT"
], [
    "blue", "green", "red", "nir", "swir1", "swir2", "lst", "QA_PIXEL", "QA_RADSAT"
])

ls7: ee.ImageCollection = ee.ImageCollection("LANDSAT/LE07/C02/T1_L2")
ls7 = ls7.select([
    "SR_B1", "SR_B2", "SR_B3", "SR_B4", "SR_B5", "SR_B7", "ST_B6", "QA_PIXEL", "QA_RADSAT"
], [
    "blue", "green", "red", "nir", "swir1", "swir2", "lst", "QA_PIXEL", "QA_RADSAT"
])

ls5: ee.ImageCollection = ee.ImageCollection("LANDSAT/LT05/C02/T1_L2")
ls5 = ls5.select([
    "SR_B1", "SR_B2", "SR_B3", "SR_B4", "SR_B5", "SR_B7", "ST_B6", "QA_PIXEL", "QA_RADSAT"
], [
    "blue", "green", "red", "nir", "swir1", "swir2", "lst", "QA_PIXEL", "QA_RADSAT"
])

ls4: ee.ImageCollection = ee.ImageCollection("LANDSAT/LT04/C02/T1_L2")
ls4 = ls4.select([
    "SR_B1", "SR_B2", "SR_B3", "SR_B4", "SR_B5", "SR_B7", "ST_B6", "QA_PIXEL", "QA_RADSAT"
], [
    "blue", "green", "red", "nir", "swir1", "swir2", "lst", "QA_PIXEL", "QA_RADSAT"
])

landsat: ee.ImageCollection = ls9.merge(ls8).merge(ls7).merge(ls5).merge(ls4)

# Further filtering and processing of Landsat data

* Remove images from non-summer months
  * Also remove meteorological data for non-summer months
* Mask bad pixels
  * Clouds
  * Water
  * Saturated pixels
* Transform digital numbers to real values
* Encode time as a band

In [3]:
# Only include data from June, July, and August
landsat = landsat.filter(
    ee.Filter.calendarRange(6, 8, "month")
)

# Only include met data from June, July, and August
met_data = met_data.filter(
    ee.Filter.rangeContains("Month", 6, 8)
)

# Mask pixels with clouds or water, or saturated pixels
def _apply_mask(image: ee.Image) -> ee.Image:
    # Bit 0: Fill
    # Bit 1: Dilated Cloud
    # Bit 2: Cirrus (high confidence)
    # Bit 3: Cloud
    # Bit 4: Cloud Shadow
    # Bit 7: Water
    qa_mask = image.select("QA_PIXEL").bitwiseAnd(int("10011111", 2)).eq(0)

    saturation_mask = image.select("QA_RADSAT").eq(0)

    return image.updateMask(qa_mask).updateMask(saturation_mask)

landsat = landsat.map(_apply_mask)

# Scale Landsat data to useful measurements
def _scale_landsat(image: ee.Image) -> ee.Image:
    optical_bands = image.select("blue", "green", "red", "nir", "swir1", "swir2").multiply(0.0000275).add(-0.2)
    thermal_bands = image.select("lst").multiply(0.00341802).add(149.0)

    return image.addBands(optical_bands, None, True).addBands(
        thermal_bands,
        None,
        True
    )

landsat = landsat.map(_scale_landsat)


# Add time band
landsat = landsat.map(
    lambda image: image.addBands(
        ee.Image.constant(image.get("system:time_start")).rename("time")
    )
)

# Remove QA bands
landsat = landsat.select(
    landsat.first().bandNames().filter(
        ee.Filter.And(
            ee.Filter.neq("item", "QA_PIXEL"),
            ee.Filter.neq("item", "QA_RADSAT")
        )
    )
)

# Generate Samples

Every case study is associated with a set of transect points. Every transect point is associated with several samples in time. Each sample contains relevant data for a specific point in time.

## Output Data Columns

* name (str)
* start (int)
* end (int)
* similarity (float)
* distance (int)
* time (int)
* station (str)
* station_temp (float)
* station_humidity (float)
* blue (float)
* green (float)
* red (float)
* nir (float)
* swir1 (float)
* swir2 (float)

In [ ]:
def map_study(study: ee.Feature) -> ee.FeatureCollection:

  study_points: ee.FeatureCollection = points.filter(
      ee.Filter.eq("name", study.get("name"))
  )

  # Only include images containing a sample point
  images: ee.ImageCollection = landsat.filterBounds(study_points.bounds())

  # Only include Landsat data between 5 years before and after implementation
  start_year = ee.Number(study.get("start")).add(-5)
  end_year = ee.Number(study.get("end")).add(6)

  images = images.filterDate(
      ee.String(start_year),
      ee.String(end_year)
  )

  # Augment points with study data
  study_points = study_points.map(
      lambda point: ee.Feature(point).set({
          "start": study.get("start"),
          "end": study.get("end"),
          "similarity": study.get("similarity")
      })
  )

  # For each image, take samples at all points
  study_samples: ee.FeatureCollection = images.map(
      lambda image: ee.Image(image).reduceRegions(
          collection = study_points,
          reducer = ee.Reducer.first(), # A point shouldn't intersect multiple pixels
          scale = 30, # meters
          crs = ee.Projection("EPSG:4326")
      )
  ).flatten()

  # Remove samples on masked pixels
  study_samples = study_samples.filter(ee.Filter.neq("time", None))

  # Associate samples with meteorological station data
  station_data: ee.FeatureCollection = met_data.filter(
      ee.Filter.eq("STATION", study.get("station"))
  )

  def get_met_data(sample: ee.Feature) -> ee.Feature:

    # Get met data for specific sample in time
    study_time = ee.Date(sample.get("time"))
    year = study_time.get("year")
    month = study_time.get("month")
    day = study_time.get("day")
    hour = study_time.get("hour")

    data = station_data.filter(
        ee.Filter.And(
            ee.Filter.eq("Year", year),
            ee.Filter.eq("Month", month),
            ee.Filter.eq("Day", day),
            ee.Filter.eq("Hour", hour),
        )
    ).first()

    # Exclude sample if no met data available at same time

    return ee.Algorithms.If(
        data,
        sample.set({
            "station_temp": data.get("temperature"),
            "station_humidity": data.get("relative_humidity")
            }),
        None
    )

  study_samples = study_samples.map(get_met_data, dropNulls=True)

  # Remove incomplete met data
  study_samples = study_samples.filter(
      ee.Filter.And(
          ee.Filter.neq("station_temp", None),
          ee.Filter.neq("station_humidity", None)
      )
  )

  return study_samples

# Run mapping function to generate samples
all_samples: ee.FeatureCollection = studies.map(map_study).flatten()

# Export

The resulting feature collection will be exported as a Google Earth Engine table asset. An export task will be created and started.

In [ ]:
task = ee.batch.Export.table.toAsset(
    collection = all_samples,
    assetId = export_path,
    description = "ExportSamples"
)

task.start()
print(task.status())

{'state': 'READY', 'description': 'ExportSamples', 'priority': 100, 'creation_timestamp_ms': 1775342290568, 'update_timestamp_ms': 1775342290568, 'start_timestamp_ms': 0, 'task_type': 'EXPORT_FEATURES', 'id': 'H5ZJ7FKDH3CWEIO5WHPF2L5V', 'name': 'projects/gee-481701/operations/H5ZJ7FKDH3CWEIO5WHPF2L5V'}
